# Explore STAC results with Cuiman

A STAC **Item** describes a scene: where it is, when it was captured, and which files belong to it. Those files are its **assets**. An **ItemCollection** groups several scenes.

In this walkthrough you will run a process, browse its scenes on a map with preview images, and open a data asset as an xarray Dataset. Then you will repeat the same steps for a process that returns links to STAC metadata.

Before starting, run `pixi run serve` in a separate terminal at the repository root and use the Pixi Python kernel here. Run the notebook and service on the same computer for this local demo. Each submission creates a new job; rerunning a display or read cell uses the existing job.

The scenes have geographic footprints in central Germany and small synthetic datasets. Their colours and values are for learning, not scientific interpretation.

In [ ]:
import pystac
import xarray as xr

from cuiman import Client
from gavicore.models import ProcessRequest

## Connect to the processing service

Create a client for the local service. For your own service, use its URL and authentication settings.

In [ ]:
client = Client(
    api_url="http://127.0.0.1:8008",
    auth={"auth_type": "none"},
)

## 1. Run a process and browse its scene

This process returns STAC metadata directly in its results. Request two scenes and keep the returned job ID.

In [ ]:
inline_job = client.execute_process(
    "create_inline_stac", request=ProcessRequest(inputs={"item_count": 2})
)
inline_job.jobID

### Show the assets

Open the `item` output as a PySTAC Item. Cuiman waits for the job to finish. `show_assets()` displays the scene's footprint, a preview, and a table of assets.

Look for **NDVI grid** with role `data`. The **Synthetic scene preview** is a picture for browsing, and **Scene summary** is a CSV report. The copy button beside an asset copies a Python expression you can use to select it.

In [ ]:
inline_item = client.open_job_result(
    inline_job.jobID, output_name="item", data_type=pystac.Item
)
client.show_assets(inline_item)

### Open the data asset

Select the `data` Asset from the Item and pass it to `open_job_result()`. `data_type=xr.Dataset` chooses the dataset reader, and `engine="zarr"` specifies the format. You do not need to extract or rewrite the asset's URL.

In [ ]:
inline_dataset = client.open_job_result(
    inline_item.assets["data"], data_type=xr.Dataset, engine="zarr"
)
inline_dataset

The Dataset lists its variables and coordinates. Access the `ndvi` variable's values to read this small grid into memory. You should see `[[0, 1], [2, 3]]`.

In [ ]:
inline_dataset["ndvi"].values

Close a Dataset when you have finished using it, to release its resources.

In [ ]:
inline_dataset.close()

## 2. Run the process that returns linked metadata

This process returns URLs pointing to STAC documents. Your workflow stays the same: keep the job ID and ask Cuiman to open the output you want.

In [ ]:
linked_job = client.execute_process(
    "create_linked_stac", request=ProcessRequest(inputs={"item_count": 2})
)
linked_job.jobID

### Browse all scenes

This time, open the `item_collection` output as an ItemCollection. Cuiman fetches its metadata and `show_assets()` displays each scene with its footprint, preview, and asset table. The scenes cover different locations.

In [ ]:
linked_items = client.open_job_result(
    linked_job.jobID, output_name="item_collection", data_type=pystac.ItemCollection
)
client.show_assets(linked_items)

### Open a scene's data asset

Choose the second scene with `.items[1]`, then select its `data` Asset. Asset reading works the same way whether the process returned embedded or linked metadata.

In [ ]:
linked_dataset = client.open_job_result(
    linked_items.items[1].assets["data"], data_type=xr.Dataset, engine="zarr"
)
linked_dataset

Read the second scene's values. This grid starts at 1, so you should see `[[1, 2], [3, 4]]`.

In [ ]:
linked_dataset["ndvi"].values

## Finish

Close the Dataset and client. To explore again, reuse the scene objects above or submit a new job. The local service retains its generated products under `.pixi/testing-stac`; stop the service when you are finished.

In [ ]:
linked_dataset.close()
client.close()